In [1]:
import pandas as pd
import numpy as np

In [2]:
hosp_path = '/project2/pishgar_1407/anijjar/mimic-iv-1.0/hosp/'
icu_path  = '/project2/pishgar_1407/anijjar/mimic-iv-1.0/icu/'
core_path = '/project2/pishgar_1407/anijjar/mimic-iv-1.0/core/'

In [3]:
data = pd.read_csv(core_path + 'patients.csv.gz')
data2 = pd.read_csv(icu_path + 'icustays.csv.gz')


#patients admitting in SICU
data_sicu = data2[data2['first_careunit']=='Surgical Intensive Care Unit (SICU)']
#data_sicu = data_sicu[data_sicu['los'] <= 2.0].copy()

#sorting them and keeping only first time admits
data_sicu_sort = data_sicu.sort_values(by=['subject_id', 'intime']).drop_duplicates(subset='subject_id', keep='first')

#Merging the patients and icustays
data_merge = data.merge(data_sicu_sort, on='subject_id')

In [4]:
df = pd.read_csv(core_path + 'admissions.csv.gz')
df['insurance'].value_counts()

insurance
Other       301530
Medicare    171360
Medicaid     50850
Name: count, dtype: int64

In [5]:
#calculating age using anchorage and anchor_year
data_merge['intime'] = pd.to_datetime(data_merge['intime'], errors='coerce')
data_merge['anchor_year'] = data_merge['anchor_year'].astype(int)

data_merge['age'] = data_merge['anchor_age'] + ((data_merge['intime']).dt.year - data_merge['anchor_year'])

#Excluding patients who are <55 years
data_final = data_merge[data_merge['age'] >= 55].copy()

#getting all hospital admission id's of the patients
hadm_code = set(data_final["hadm_id"].tolist())

In [6]:
diag = pd.read_csv(hosp_path + 'diagnoses_icd.csv.gz')

diag = diag[diag["hadm_id"].isin(hadm_code)].copy()
diag = diag[diag["seq_num"]==1].copy()

# ICD-9 and ICD-10 prefixes for stroke
stroke_icd9_prefixes = ['430','431','433','434','435', '436']
stroke_icd10_prefixes = ['I60','I61','I63','I64']

diag['icd_code'] = diag['icd_code'].astype(str).str.upper()
diag['icd_version'] = diag['icd_version'].astype('Int64')

# Identify stroke rows
v9 = ((diag['icd_version'] == 9) & (diag['icd_code'].str[:3].isin(stroke_icd9_prefixes)))
v10 =((diag['icd_version'] == 10) & (diag['icd_code'].str[:3].isin(stroke_icd10_prefixes)))

diag['is_stroke'] = v9 | v10

In [7]:
#data distribution
diag['is_stroke'].value_counts()

is_stroke
False    5772
True     1355
Name: count, dtype: Int64

In [8]:
#Getting the unique subject_id of the stroke patients
stroke = diag.loc[diag['is_stroke'], 'subject_id'].unique()
print(f"Shape of the stroke patients: {stroke.shape}")

#Stating each patient whether they are stroke or not stroke
data_final['is_stroke'] = data_final['subject_id'].isin(stroke).astype(np.int8)
data_final.head()

Shape of the stroke patients: (1355,)


,subject_id,gender,anchor_age,anchor_year,anchor_year_group,dod,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,age,is_stroke
1,10722531,F,59,2161,2014 - 2016,NaN,20223681,30357477,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2161-04-15 14:29:03,2161-04-19 00:44:40,3.427512,59,0
2,11062755,M,76,2174,2008 - 2010,NaN,23407263,38273332,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2174-10-25 23:45:00,2174-10-29 20:39:23,3.871100,76,1
4,11178069,F,62,2178,2008 - 2010,NaN,28499612,37173753,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2184-07-19 19:33:00,2184-07-22 19:07:51,2.982535,68,0
7,11864106,F,59,2184,2008 - 2010,NaN,24588665,35824540,Surgical Intensive Care Unit (SICU),Trauma SICU (TSICU),2188-02-09 13:12:29,2188-02-10 16:39:51,1.144005,63,0
8,12001881,M,68,2114,2011 - 2013,NaN,25409877,30302331,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2114-04-03 03:01:39,2114-04-03 15:13:52,0.508484,68,0


In [9]:
#Distribution of stroke patients
print(data_final['is_stroke'].value_counts())

print("Unique values of hadm_id: ", data_final['hadm_id'].nunique())
print("Unique values of subject_id: ", data_final['subject_id'].nunique())

is_stroke
0    5774
1    1355
Name: count, dtype: int64
Unique values of hadm_id:  7129
Unique values of subject_id:  7129


In [10]:
cohort = data_final.copy()

print(f"Null values of the dataset: \n{cohort.isnull().sum()}")



Null values of the dataset: 
subject_id              0
gender                  0
anchor_age              0
anchor_year             0
anchor_year_group       0
dod                  5705
hadm_id                 0
stay_id                 0
first_careunit          0
last_careunit           0
intime                  0
outtime                 0
los                     0
age                     0
is_stroke               0
dtype: int64


In [11]:
patients   = pd.read_csv(core_path + 'patients.csv.gz', usecols=['subject_id','gender','anchor_age'])
admissions = pd.read_csv(core_path + 'admissions.csv.gz', usecols=['subject_id','hadm_id','ethnicity', 'insurance'])
icustays   = pd.read_csv(icu_path  + 'icustays.csv.gz', usecols=['subject_id','hadm_id','stay_id','intime'])

# Compute BMI
#icustays['BMI'] = icustays['weight'] / (icustays['height']/100)**2
# Restrict icustays to cohort patients
#icustays = icustays[icustays['stay_id'].isin(cohort['stay_id'])]
#icustays_short = icustays[['subject_id','hadm_id','stay_id','intime']].copy()
#icustays_short['intime'] = pd.to_datetime(icustays_short['intime'])

icustays['intime'] = pd.to_datetime(icustays['intime'])

# Merge demographic info
demo = (patients.merge(admissions, on='subject_id', how='left')
                .merge(icustays[['subject_id','hadm_id','stay_id']], on=['subject_id','hadm_id'], how='left'))


In [12]:
diagnoses = pd.read_csv(hosp_path + 'diagnoses_icd.csv.gz',
                        usecols=['subject_id','hadm_id','icd_code','icd_version'])

# Define ICD prefixes (short list)
comorb_prefixes = {
    'pvd': (['440','441','442','443','444'], ['I70','I71','I72','I73']),
    'htn': (['401','402','403','404','405'], ['I10','I11','I12','I13','I15']),
    'copd': (['490','491','492','493','494','495','496'], ['J40','J41','J42','J43','J44']),
    'diabetes': (['250'], ['E10','E11','E12','E13','E14']),
    'renal': (['585'], ['N18']),
    'liver': (['570','571','572','573'], ['K70','K71','K72','K73','K74','K75','K76','K77']),
    'pud': (['531','532','533','534'], ['K25','K26','K27','K28']),
    'sepsis': (['038'], ['A40','A41']),
    'chf': (['428'], ['I50']),
    'cancer': ([str(x) for x in range(140,240)], ['C']),
    'rheumatic': (['710'], ['M05','M06'])
}
diagnoses['icd_code'] = diagnoses['icd_code'].astype(str).str.upper()
diagnoses['icd_version'] = diagnoses['icd_version'].astype('Int64')



In [13]:
for name, (icd9, icd10) in comorb_prefixes.items():
    diagnoses[name] = np.where(
        ((diagnoses['icd_version']==9) & diagnoses['icd_code'].str[:3].isin(icd9)) |
        ((diagnoses['icd_version']==10) & diagnoses['icd_code'].str[:3].isin(icd10)),
        1, 0)
comorb = diagnoses.groupby(['subject_id','hadm_id'])[list(comorb_prefixes.keys())].max().reset_index()
diagnoses

,subject_id,hadm_id,icd_code,icd_version,pvd,htn,copd,diabetes,renal,liver,pud,sepsis,chf,cancer,rheumatic
0,15734973,20475282,2825,9,0,0,0,0,0,0,0,0,0,0,0
1,15734973,20475282,V0251,9,0,0,0,0,0,0,0,0,0,0,0
2,15734973,20475282,V270,9,0,0,0,0,0,0,0,0,0,0,0
3,15734973,20475282,64891,9,0,0,0,0,0,0,0,0,0,0,0
4,15734973,20475282,66481,9,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5280346,13747041,25594844,R531,10,0,0,0,0,0,0,0,0,0,0,0
5280347,13747041,25594844,R0902,10,0,0,0,0,0,0,0,0,0,0,0
5280348,13747041,25594844,F1120,10,0,0,0,0,0,0,0,0,0,0,0
5280349,13747041,25594844,J189,10,0,0,0,0,0,0,0,0,0,0,0


In [14]:
comorb

,subject_id,hadm_id,pvd,htn,copd,diabetes,renal,liver,pud,sepsis,chf,cancer,rheumatic
0,10000019,25058216,0,0,0,0,0,0,0,0,0,0,0
1,10000032,22595853,0,0,1,0,0,1,0,0,0,0,0
2,10000032,22841357,0,0,1,0,0,1,0,0,0,0,0
3,10000032,25742920,0,0,1,0,0,1,0,0,0,0,0
4,10000032,29079034,0,0,1,0,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
521106,19999828,25744818,0,1,1,1,0,0,0,0,0,0,0
521107,19999828,29734428,0,1,1,1,0,0,0,0,0,0,0
521108,19999840,21033226,0,1,0,0,0,0,0,0,0,0,0
521109,19999840,26071774,0,1,0,0,0,0,0,0,0,0,0


In [15]:
diagn = pd.read_csv(hosp_path + 'diagnoses_icd.csv.gz',
                        usecols=['subject_id','hadm_id','icd_code','icd_version'])
addictives={
    'Alcohol': (['30300', '30500', 'V113'],['F10']), #	Alcohol abuse or dependence
    'Tobacco': (['30501'], ['F17']), #	Tobacco use disorder
    'Opioid': (['30400', '30550'],['F11']), #	Opioid dependence or use
    'Cannabis': (['30430'],['F12']), #	Cannabis use
    'Cocaine': (['30420', '30560'],['F14']), #	Cocaine use
    'Sedative': (['30410', '30540'],['F13']), #	Sedative use disorder
    'Other': (['30490', '30590'],['F19']),
    'Smoking': (['Z87891'], ['3051'])
}

diagn['icd_code'] = diagn['icd_code'].astype(str).str.upper()
diagn['icd_version'] = diagn['icd_version'].astype('Int64')


#diagn[diagn['icd_code']=='']#.str.contains("alcohol abuse", case=False, na=False)].sort_values('icd_code')

In [16]:
for name, (icd9, icd10) in addictives.items():
    diagn[name] = np.where(
        ((diagn['icd_version']==9) & diagn['icd_code'].str[:3].isin(icd9)) |
        ((diagn['icd_version']==10) & diagn['icd_code'].str[:3].isin(icd10)),
        1, 0)
addict = diagn.groupby(['subject_id','hadm_id'])[list(addictives.keys())].max().reset_index()
addict

,subject_id,hadm_id,Alcohol,Tobacco,Opioid,Cannabis,Cocaine,Sedative,Other,Smoking
0,10000019,25058216,0,0,0,0,0,0,0,0
1,10000032,22595853,0,0,0,0,0,0,0,0
2,10000032,22841357,0,0,0,0,0,0,0,0
3,10000032,25742920,0,0,0,0,0,0,0,0
4,10000032,29079034,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...
521106,19999828,25744818,0,0,0,0,0,0,0,0
521107,19999828,29734428,0,0,0,1,0,0,0,0
521108,19999840,21033226,0,0,0,0,0,0,0,0
521109,19999840,26071774,0,0,0,0,0,0,0,0


In [17]:
lab_ids = [50931,50868,50862,50885,50912,50810,50811,51301,50813,
           51265,50971,51275,51274,51237,51006, 50970, 50983, 
           50960, 50902, 50893, 50889, 50878, 51003]

icustays = icustays[icustays['stay_id'].isin(cohort['stay_id'])]
icustays_short = icustays[['subject_id','hadm_id','stay_id','intime']].copy()
icustays_short['intime'] = pd.to_datetime(icustays_short['intime'])


In [18]:
chunks = []
cohort_hadm = set(cohort['hadm_id'])
for chunk in pd.read_csv(hosp_path+'labevents.csv.gz',
                         usecols=['subject_id','hadm_id','itemid','charttime','valuenum'],
                         chunksize=500000):
    chunk = chunk[chunk['hadm_id'].isin(cohort_hadm)] 
    chunk = chunk[chunk['itemid'].isin(lab_ids)]
    chunk = chunk.merge(icustays_short, on=['subject_id','hadm_id'], how='inner')
    chunk['charttime'] = pd.to_datetime(chunk['charttime'])
    mask = (chunk['charttime'] >= chunk['intime']) & (chunk['charttime'] <= chunk['intime'] + pd.Timedelta(hours=24))
    chunk = chunk.loc[mask]
    #agg = chunk.groupby(['stay_id','itemid'])['valuenum'].agg(['min','max','mean']).reset_index()
    chunks.append(chunk)

lab_summary = pd.concat(chunks, ignore_index=True)
print("Labs processed:", len(lab_summary))

Labs processed: 216648


In [19]:
labs_summary = (
    lab_summary.pivot_table(
        index=['stay_id'],
        columns='itemid',
        values='valuenum',
        aggfunc=['mean', 'max', 'min']
    ).reset_index()
)


In [20]:
labs_summary.rename(columns={
    50810: 'Hematocrit',
    50811: 'Hemoglobin',
    50813: 'Lactate',
    50862: 'Albumin',
    50868: 'Anion Gap',
    50885: 'Bilirubin',
    50912: 'Creatinine',
    50931: 'Glucose',
    50971: 'Potassium',
    51006: 'Urea Nitrogen',
    51237: 'INR(PT)',
    51265: 'Platelet Count',
    51274: 'PT',
    51275: 'PTT',
    51301: 'WBC',
    50970: 'Phosphate',
    50983: 'Na',
    50960: 'Mg',
    50902: 'chloride',
    50893: 'Ca',
    50889: 'CRP',
    50878: 'AST',
    51003: 'Troponin' 
}, inplace=True)

In [21]:
labs_summary.columns = ['stay_id'] + [
    f"{col[0]}_{col[1]}" if isinstance(col, tuple) and col[1] else col[0]
    for col in labs_summary.columns[1:]
]


In [22]:
vital_ids = [220045,220179,220180,220181,220277]
#height,weight=226707,226512
chunks = []

for chunk in pd.read_csv(icu_path+'chartevents.csv.gz',
                         usecols=['subject_id','hadm_id','itemid','charttime','valuenum'],
                         chunksize=500000):
    chunk = chunk[chunk['hadm_id'].isin(cohort_hadm)]
    chunk = chunk[chunk['itemid'].isin(vital_ids)]
    chunk = chunk.merge(icustays_short, on=['subject_id','hadm_id'], how='inner')
    chunk['charttime'] = pd.to_datetime(chunk['charttime'])
    mask = (chunk['charttime'] >= chunk['intime']) & (chunk['charttime'] <= chunk['intime'] + pd.Timedelta(hours=24))
    chunk = chunk.loc[mask]
    #agg = chunk.groupby(['stay_id','itemid'])['valuenum'].agg(['min','max','mean']).reset_index()
    chunks.append(chunk)

vital_summary = pd.concat(chunks, ignore_index=True)
print("Vitals processed:", len(vital_summary))


Vitals processed: 735964


In [23]:
vitals_summary = (
    vital_summary.pivot_table(
        index=['stay_id'],
        columns='itemid',
        values='valuenum',
        aggfunc=['mean', 'max', 'min']
    ).reset_index()
)

In [24]:
vitals_summary.rename(columns={
    220045: 'Heart Rate',
    220179: 'systolic_BP',
    220180: 'diastolic_BP',
    220181: 'mean_BP',
    220277: 'Spo2'
}, inplace=True)

In [25]:
vitals_summary.columns = ['stay_id'] + [
    f"{col[0]}_{col[1]}" if isinstance(col, tuple) else col
    for col in vitals_summary.columns[1:]
]


In [26]:
press = pd.read_csv(hosp_path + 'prescriptions.csv.gz', low_memory=False)

press['drug'] = press['drug'].str.lower()

drug_classes = {
    'anticoagulant': ['warfarin','heparin','enoxaparin','apixaban','rivaroxaban','dabigatran'],
    'aspirin': ['aspirin','clopidogrel','ticagrelor','prasugrel'],
    'statin': ['atorvastatin','simvastatin','rosuvastatin','pravastatin','lovastatin','pitavastatin'],
    'insulin': ['insulin','metformin','glipizide','glyburide','sitagliptin','liraglutide']
}

# Create flags
for cls, keywords in drug_classes.items():
    pattern = '|'.join(keywords)
    press[cls] = press['drug'].str.contains(pattern, case=False, na=False).astype(int)

In [27]:
proced = pd.read_csv(hosp_path + 'procedures_icd.csv.gz')

first_op = proced.groupby('hadm_id')['chartdate'].min().reset_index().rename(columns={'chartdate':'op_time'})

# Merge and filter prescriptions before operation
presc = press.merge(first_op, on='hadm_id', how='left')
presc_preop = presc[presc['starttime'] < presc['op_time']]


In [28]:
preop_flags = (
    presc_preop.groupby(['subject_id','hadm_id'])
               [['anticoagulant','aspirin','statin','insulin']]
               .max()
               .reset_index()
)
preop_flags

,subject_id,hadm_id,anticoagulant,aspirin,statin,insulin
0,10000019,25058216,0,0,0,0
1,10000032,22841357,1,0,0,0
2,10000117,27988844,1,0,0,0
3,10000200,24930234,0,0,0,0
4,10000560,28979390,0,0,0,0
...,...,...,...,...,...,...
104730,19999464,23033573,1,0,0,0
104731,19999784,23664472,0,0,0,0
104732,19999784,24755486,1,0,0,0
104733,19999784,29956342,1,0,0,0


In [29]:
final = (cohort
         .merge(demo, on=['subject_id','hadm_id','stay_id'], how='left', validate='one_to_one')
         .merge(comorb, on=['subject_id','hadm_id'], how='left', validate='one_to_one')
         .merge(addict, on=['subject_id','hadm_id'], how='left', validate='one_to_one')
         .merge(preop_flags, on=['subject_id','hadm_id'], how='left', validate='one_to_one')
         .merge(labs_summary, on='stay_id', how='left')
         .merge(vitals_summary, on='stay_id', how='left'))


In [30]:
final = final.drop_duplicates('stay_id')
final.to_csv('mimic_iv_data.csv', index=False)
print("Final shape:", final.shape)
#final

Final shape: (7129, 126)


In [31]:
print("Cohort:", 'is_stroke' in final.columns)
final.isnull().sum()

Cohort: True


subject_id             0
gender_x               0
anchor_age_x           0
anchor_year            0
anchor_year_group      0
                    ... 
min_Heart Rate        18
min_systolic_BP      688
min_diastolic_BP     689
min_mean_BP          688
min_Spo2              25
Length: 126, dtype: int64